# Portfolio Exercise - Compare Recommender Strategies with MovieLens

In the lecture, you already implemented simple personalized recommender systems. In this portfolio exercise, the goal is **not to rebuild every algorithm from scratch**.

Instead, you will investigate a more realistic question:

> **How do different recommender strategies behave, and how should we evaluate them?**

You will compare three approaches:

1. **Popularity-based recommendation** - global interaction counts
2. **Content-based filtering** - similarity between a user profile and movie features
3. **User-based collaborative filtering** - similarity between users' rating behaviour

The content-based and collaborative-filtering implementations are provided. You will still write Python to inspect and transform data, build the baseline, compare rankings, evaluate the systems, and run a controlled experiment.

## What you should demonstrate

By the end of the notebook, you should be able to:

- identify users, items and interactions
- explain what evidence each recommender uses
- generate and compare Top-K rankings
- interpret **Precision@K** and **Recall@K**
- explain why chronological evaluation is preferable to a random split here
- reason about cold start, popularity bias and feedback loops

> **AI use is allowed.** You may use AI, documentation and your lecture material. During the **portfolio defence**, however, you must be able to explain your code, the recommendation results, the evaluation setup, and what you expect to happen when the data changes.


## What to submit

Submit **this completed notebook** in your portfolio. It should run from top to bottom after you complete the TODOs.

Keep written answers short and concrete. This is a technical portfolio artefact, not a report.


# 0. Dataset and setup

We use **MovieLens latest-small** from GroupLens. Ratings use a **0.5–5.0** scale and include timestamps. Movie information includes titles and genres.

The setup cell downloads the dataset the first time you run it.


In [ ]:
from pathlib import Path
import urllib.request
import zipfile

import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

pd.set_option("display.max_colwidth", 90)
pd.set_option("display.precision", 3)

DATA_URL = "https://files.grouplens.org/datasets/movielens/ml-latest-small.zip"
DATA_DIR = Path("data")
ZIP_PATH = DATA_DIR / "ml-latest-small.zip"
EXTRACTED_DIR = DATA_DIR / "ml-latest-small"

DATA_DIR.mkdir(exist_ok=True)

if not EXTRACTED_DIR.exists():
    if not ZIP_PATH.exists():
        print("Downloading MovieLens latest-small...")
        urllib.request.urlretrieve(DATA_URL, ZIP_PATH)
    print("Extracting dataset...")
    with zipfile.ZipFile(ZIP_PATH, "r") as zf:
        zf.extractall(DATA_DIR)

print("Dataset ready:", EXTRACTED_DIR)


In [ ]:
ratings = pd.read_csv(EXTRACTED_DIR / "ratings.csv")
movies = pd.read_csv(EXTRACTED_DIR / "movies.csv")
ratings["datetime"] = pd.to_datetime(ratings["timestamp"], unit="s")

print(f"Ratings: {len(ratings):,}")
print(f"Users:   {ratings['userId'].nunique():,}")
print(f"Movies:  {movies['movieId'].nunique():,}")
print(f"Period:  {ratings['datetime'].min().date()} to {ratings['datetime'].max().date()}")

display(ratings.head())
display(movies.head())


# 1. Create a chronological train/test split

A recommender should learn from **past behaviour** and be evaluated on **later behaviour**.

For every user, we therefore:

- sort interactions by timestamp
- use the earliest **80%** as training data
- use the latest **20%** as test data

This is a **per-user chronological split**. It is more realistic than randomly hiding ratings because later interactions for a user are not placed in that user's training history.

> For simplicity, this is **not one global time cut-off** across the complete dataset. Different users can therefore have different calendar-time split points.

Do not change the split code.

In [ ]:
ratings_sorted = ratings.sort_values(
    ["userId", "timestamp", "movieId"]
).reset_index(drop=True)

ratings_sorted["interaction_nr"] = ratings_sorted.groupby("userId").cumcount()
ratings_sorted["n_user_interactions"] = ratings_sorted.groupby("userId")["movieId"].transform("size")
ratings_sorted["split_index"] = np.floor(
    0.80 * ratings_sorted["n_user_interactions"]
).astype(int)


train_mask = ratings_sorted["interaction_nr"] < ratings_sorted["split_index"]

ratings_train = (
    ratings_sorted.loc[train_mask, ratings.columns]
    .copy()
    .reset_index(drop=True)
)
ratings_test = (
    ratings_sorted.loc[~train_mask, ratings.columns]
    .copy()
    .reset_index(drop=True)
)

KNOWN_MOVIE_IDS = set(ratings_train["movieId"].unique())

print(f"Training interactions: {len(ratings_train):,}")
print(f"Test interactions:     {len(ratings_test):,}")
print(f"Movies known in train: {len(KNOWN_MOVIE_IDS):,}")


## Select users for the exercise

For evaluation, a test movie is considered **relevant** when the user rated it **4.0 or higher**.

To keep comparisons meaningful, we evaluate only test movies that already occur somewhere in the training data. This gives all three recommenders the same existing-item candidate universe; we discuss the **new-item cold-start** case separately later.

The code below selects users with enough history for the exercise. Choose one displayed `userId` as your `TARGET_USER`.


In [ ]:
train_movie_counts = ratings_train.groupby("movieId").size()
popular_train_movie_ids = set(train_movie_counts[train_movie_counts >= 20].index)

train_counts = ratings_train.groupby("userId").size()
train_liked_popular = ratings_train[
    (ratings_train["rating"] >= 4.0)
    & ratings_train["movieId"].isin(popular_train_movie_ids)
]
train_liked_popular_counts = train_liked_popular.groupby("userId").size()
train_five_star_counts = ratings_train[
    ratings_train["rating"] == 5.0
].groupby("userId").size()

relevant_test_known = ratings_test[
    (ratings_test["rating"] >= 4.0)
    & ratings_test["movieId"].isin(KNOWN_MOVIE_IDS)
]
relevant_test_counts = relevant_test_known.groupby("userId").size()

eligible_users = pd.DataFrame({
    "train_interactions": train_counts,
    "liked_popular_train": train_liked_popular_counts,
    "relevant_test_items": relevant_test_counts,
    "five_star_train": train_five_star_counts,
}).fillna(0).astype(int)

eligible_users = eligible_users[
    (eligible_users["train_interactions"] >= 50)
    & (eligible_users["liked_popular_train"] >= 10)
    & (eligible_users["relevant_test_items"] >= 5)
    & (eligible_users["five_star_train"] >= 1)
]

if eligible_users.empty:
    raise RuntimeError(
        "No eligible users found. Check that MovieLens latest-small loaded correctly."
    )

eligible_sample = eligible_users.sample(
    min(15, len(eligible_users)), random_state=42
).sort_values("train_interactions", ascending=False)

display(eligible_sample)


In [ ]:
# TODO Choose one userId from the table above.
# The default is valid, but you may select another displayed user.
TARGET_USER = int(eligible_sample.index[0])

print("Target user:", TARGET_USER)

# 2. Understand your target user

Work only with the user's **training history**: these are the interactions the recommender is allowed to know.

Create `target_history` containing at least:

- `movieId`
- `title`
- `genres`
- `rating`
- `datetime`

Sort the user's highest ratings first.


In [ ]:
# TODO: create the target user's training history.
# Hint: filter ratings_train, merge with movies, select useful columns, then sort.
target_history = ...

display(target_history.head(30))


In [ ]:
# Create the target user's training history sorted by highest rating
target_history = (
    ratings_train[ratings_train["userId"] == TARGET_USER]
    .merge(movies[["movieId", "title", "genres"]], on="movieId", how="left")
    [["movieId", "title", "genres", "rating", "datetime"]]
    .sort_values(by=["rating", "datetime"], ascending=[False, False])
    .reset_index(drop=True)
)

display(target_history.head(30))

### Questions 1-2

**1. For MovieLens, identify:**

- User: ...
- Item: ...
- Interaction: ...
- Explicit or implicit feedback? ...

**2. Based on the training history above, describe one apparent preference of this user.** Refer to actual titles, genres or rating patterns.

**YOUR ANSWER (2-4 sentences):**  
...

# MovieLens Analysis

## 1. Dataset Entities and Feedback Type

**User:** An individual viewer represented by a unique `userId`.

**Item:** A movie represented by a unique `movieId` (associated with a title and genres).

**Interaction:** A rating event where a user rates a movie on a scale of 0.5 to 5.0 at a recorded timestamp.

**Explicit or Implicit Feedback:** **Explicit feedback** — because users consciously and directly assign numerical star ratings expressing their satisfaction, rather than performing passive actions like clicks or watch time.

## 2. Apparent User Preference (Based on Training History)

> *Inspect your top rows in `target_history.head(30)` to reference the user's specific genres and titles.*

**Example:** The user demonstrates a strong preference for **Action** and **Sci-Fi** (or **Drama/Crime**, depending on your target user), consistently awarding **4.5 to 5.0 stars** to titles in these genres, while rating lighter comedies or romances less frequently or lower.

# 3. Build a popularity baseline

Before personalisation, create a simple baseline yourself.

For this exercise:

> **popularity score = number of ratings a movie received in `ratings_train`**

The function must:

1. count ratings per movie
2. remove movies already rated by the target user
3. add movie information
4. rank by popularity score
5. return the Top-`n` unseen movies


In [ ]:
def recommend_popularity(ratings_train, movies, target_user, n=20):
    """Recommend the most-rated unseen movies."""

    # TODO 1: count the number of ratings per movie.
    # Hint: groupby("movieId").size().rename("score").reset_index()

    # TODO 2: collect movieIds already rated by the target user.

    # TODO 3: remove seen movies, add title/genres, sort by score,
    # and return only the Top-n rows.

    # YOUR CODE HERE
    pass

In [ ]:
def recommend_popularity(ratings_train, movies, target_user, n=20):
    """Recommend the most-rated unseen movies."""

    # 1. Count the number of ratings per movie
    movie_counts = (
        ratings_train.groupby("movieId")
        .size()
        .rename("score")
        .reset_index()
    )

    # 2. Collect movieIds already rated by the target user
    seen_movie_ids = set(
        ratings_train.loc[ratings_train["userId"] == target_user, "movieId"]
    )

    # 3. Remove seen movies, add title/genres, sort by score descending, return Top-n
    unseen_movies = movie_counts[~movie_counts["movieId"].isin(seen_movie_ids)]

    ranking = (
        unseen_movies
        .merge(movies[["movieId", "title", "genres"]], on="movieId", how="left")
        .sort_values(["score", "title"], ascending=[False, True])
        .head(n)
        .reset_index(drop=True)
    )

    return ranking

In [ ]:
popularity_ranking = recommend_popularity(
    ratings_train, movies, TARGET_USER, n=20
)

display(popularity_ranking.head(5))


### Question 3 - baseline check

You removed movies the target user has already rated. Why is this recommender nevertheless **not personalised**?

**YOUR ANSWER (1-2 sentences):**  
...


The ranking score is determined entirely by global aggregate community counts across all users. Filtering out already-rated movies is merely a negative candidate filter, not personalization: the relative order of the remaining movies is identical for every single user.

# 4. Generate personalised recommendations

You already implemented personalised recommenders during the lecture exercises, so the implementations are supplied here.

We use:

- **Content-based filtering:** create a genre profile from movies the target user rated **4.0 or higher**, then rank unseen movies by **cosine similarity** to that profile.
- **User-based collaborative filtering:** find users with small **average absolute rating differences** on movies both users rated, then use those neighbours' ratings to rank unseen movies.

The collaborative method deliberately does **not** use raw cosine similarity on 1-5 ratings (the lecture showed why cosine can be misleading for this type of explicit rating data).


In [ ]:
def recommend_content_based(
    ratings_train,
    movies,
    target_user,
    n=20,
    genre_matrix=None,
    candidate_movie_ids=None,
):
    """Genre-based user profile + cosine similarity."""

    if genre_matrix is None:
        genre_matrix = movies.set_index("movieId")["genres"].str.get_dummies("|")

    target_ratings = ratings_train[ratings_train["userId"] == target_user]
    liked_ids = target_ratings.loc[target_ratings["rating"] >= 4.0, "movieId"]
    liked_ids = [movie_id for movie_id in liked_ids if movie_id in genre_matrix.index]

    if len(liked_ids) == 0:
        raise ValueError("Target user has no liked movies in the training data.")

    profile = genre_matrix.loc[liked_ids].mean(axis=0)
    user_profile = profile.to_numpy().reshape(1, -1)

    seen_ids = set(target_ratings["movieId"])

    if candidate_movie_ids is None:
        candidate_movie_ids = set(ratings_train["movieId"].unique())

    candidate_ids = [
        movie_id
        for movie_id in candidate_movie_ids
        if movie_id not in seen_ids and movie_id in genre_matrix.index
    ]

    candidate_features = genre_matrix.loc[candidate_ids]
    scores = cosine_similarity(candidate_features, user_profile).ravel()

    ranking = pd.DataFrame({
        "movieId": candidate_features.index,
        "score": scores,
    })

    ranking = (
        ranking
        .merge(movies[["movieId", "title", "genres"]], on="movieId", how="left")
        .sort_values(["score", "title"], ascending=[False, True])
        .head(n)
        .reset_index(drop=True)
    )

    return ranking, profile.sort_values(ascending=False)


In [ ]:
def recommend_user_based(
    ratings_train,
    movies,
    target_user,
    n=20,
    min_common=5,
    n_neighbours=10,
    min_support=2,
):
    """User-based CF using average absolute rating difference on common movies."""

    target = (
        ratings_train[ratings_train["userId"] == target_user]
        [["movieId", "rating"]]
        .rename(columns={"rating": "target_rating"})
    )

    if target.empty:
        raise ValueError("Target user is not present in the training data.")

    overlap = (
        ratings_train[
            (ratings_train["userId"] != target_user)
            & ratings_train["movieId"].isin(target["movieId"])
        ]
        .merge(target, on="movieId", how="inner")
    )


    overlap["abs_diff"] = (
        overlap["rating"] - overlap["target_rating"]
    ).abs()

    neighbours = (
        overlap.groupby("userId")
        .agg(
            distance=("abs_diff", "mean"),
            common_ratings=("movieId", "size"),
        )
        .reset_index()
    )

    neighbours = (
        neighbours[neighbours["common_ratings"] >= min_common]
        .sort_values(["distance", "common_ratings"], ascending=[True, False])
        .head(n_neighbours)
        .reset_index(drop=True)
    )

    if neighbours.empty:
        raise ValueError("No suitable neighbours found for this user.")

    neighbour_ids = neighbours["userId"].tolist()
    seen_ids = set(target["movieId"])

    candidate_ratings = ratings_train[
        ratings_train["userId"].isin(neighbour_ids)
        & ~ratings_train["movieId"].isin(seen_ids)
    ]

    predictions = (
        candidate_ratings.groupby("movieId")
        .agg(score=("rating", "mean"), support=("rating", "size"))
        .reset_index()
    )

    predictions = predictions[predictions["support"] >= min_support]

    ranking = (
        predictions
        .merge(movies[["movieId", "title", "genres"]], on="movieId", how="left")
        .sort_values(
            ["score", "support", "title"],
            ascending=[False, False, True],
        )
        .head(n)
        .reset_index(drop=True)
    )

    return ranking, neighbours


In [ ]:
GENRE_MATRIX = movies.set_index("movieId")["genres"].str.get_dummies("|")

content_ranking, content_profile = recommend_content_based(
    ratings_train,
    movies,
    TARGET_USER,
    n=20,
    genre_matrix=GENRE_MATRIX,
    candidate_movie_ids=KNOWN_MOVIE_IDS,
)

usercf_ranking, neighbours = recommend_user_based(
    ratings_train, movies, TARGET_USER, n=20
)

print("Content-based Top-5")
display(content_ranking.head(5))

print("Strongest genre signals in the target user's content profile")
display(content_profile.head(8).rename("profile_weight").to_frame())

print("User-based CF Top-5")
display(usercf_ranking.head(5))

print("Nearest users used by collaborative filtering")
display(neighbours.head(10))


## Compare the three Top-5 rankings

Build one table with the movie titles at ranks 1-5 for each approach (popularity, content-based and user-based CF).

Do **not** type movie names manually; construct the table from the ranking DataFrames.


In [ ]:
# TODO: build a 5-row comparison table from the three ranking DataFrames.
comparison = ...

display(comparison)


In [ ]:
# Build a 5-row comparison table showing top-5 titles from each method
comparison = pd.DataFrame({
    "Rank": range(1, 6),
    "Popularity": popularity_ranking["title"].head(5).values,
    "Content-based": content_ranking["title"].head(5).values,
    "User-based CF": usercf_ranking["title"].head(5).values,
}).set_index("Rank")

display(comparison)

In [ ]:
# Evidence that can help you explain the highest-ranked CF recommendation.
top_cf_movie_id = int(usercf_ranking.iloc[0]["movieId"])

top_cf_neighbour_evidence = (
    ratings_train[
        ratings_train["userId"].isin(neighbours["userId"])
        & (ratings_train["movieId"] == top_cf_movie_id)
    ]
    .merge(neighbours, on="userId", how="left")
    [["userId", "rating", "distance", "common_ratings"]]
    .sort_values("distance")
)

print("Neighbour ratings for the highest-ranked user-CF movie:")
display(top_cf_neighbour_evidence)


### Question 4 - compare the evidence

**a.** Does any movie appear in more than one Top-5? Give one example if so.  
...

**b.** Choose one highly ranked **content-based** recommendation. Why did it receive a high score? Refer to the user's genre profile and the movie's features.  
...

**c.** Explain why the **highest-ranked user-based CF movie** could receive a high score. Refer to the neighbours and their rating behaviour.  
...

**d.** Why can content-based filtering and collaborative filtering produce different rankings for the same user?  
...

### a. Does any movie appear in more than one Top-5?
Inspect the comparison table: if a movie repeats (e.g., a blockbuster matching the user's preferred genre), name it. If none repeat, state: "None of the movies appear in more than one Top-5 list, showing that each algorithm draws from distinct candidate pools."
### b. Content-based recommendation score explanation:
The movie's genre vector has a high cosine similarity with the user's profile. The user heavily rated movies sharing those exact genres (e.g., Drama, Sci-Fi) 4.0 or above in the training set.
### c. User-based CF highest-ranked movie explanation:
The movie received consistently high ratings from the user's nearest neighbors (users with similar rating behavior across shared movies) and met the minimum support threshold of at least 2 neighbors.
### d. Why do CBF and CF produce different rankings?
Content-based filtering relies strictly on item metadata (genres), while collaborative filtering ignores metadata entirely and recommends items based on shared rating behavior among similar users.

# 5. Evaluate the three recommenders for your target user

We now evaluate on the user's **later interactions** from `ratings_test`.

A test movie is relevant when:

- the target user rated it **4.0 or higher**; and
- the movie already existed in the training candidate universe.

We use **Top-10** recommendations and calculate:

- **Hits@10**: number of relevant test movies retrieved
- **Precision@10**: relevant items in the Top-10 / 10
- **Recall@10**: relevant items retrieved / all relevant test items


In [ ]:
def precision_at_k(recommended_ids, relevant_ids, k):
    top_k = list(recommended_ids)[:k]
    hits = sum(movie_id in relevant_ids for movie_id in top_k)
    return hits / k


def recall_at_k(recommended_ids, relevant_ids, k):
    if not relevant_ids:
        return 0.0
    top_k = list(recommended_ids)[:k]
    hits = sum(movie_id in relevant_ids for movie_id in top_k)
    return hits / len(relevant_ids)


def relevant_test_items_for_user(user_id):
    rows = ratings_test[
        (ratings_test["userId"] == user_id)
        & (ratings_test["rating"] >= 4.0)
        & (ratings_test["movieId"].isin(KNOWN_MOVIE_IDS))
    ]
    return set(rows["movieId"])


K_EVAL = 10
RELEVANT_TEST_ITEMS = relevant_test_items_for_user(TARGET_USER)
print("Relevant later test items for target user:", len(RELEVANT_TEST_ITEMS))


In [ ]:
rankings = {
    "Popularity": popularity_ranking,
    "Content-based": content_ranking,
    "User-based CF": usercf_ranking,
}

# TODO: create one row per method with Hits@10, Precision@10 and Recall@10.
rows = []

for method, ranking in rankings.items():
    recommended_ids = ...
    hits = ...
    precision = ...
    recall = ...

    rows.append({
        "method": method,
        f"Hits@{K_EVAL}": hits,
        f"Precision@{K_EVAL}": precision,
        f"Recall@{K_EVAL}": recall,
    })

target_evaluation = pd.DataFrame(rows)
display(target_evaluation)


In [ ]:
rankings = {
    "Popularity": popularity_ranking,
    "Content-based": content_ranking,
    "User-based CF": usercf_ranking,
}

rows = []

for method, ranking in rankings.items():
    # Extract the Top-K recommended movie IDs
    recommended_ids = ranking["movieId"].head(K_EVAL).tolist()

    # Calculate hits, precision, and recall
    hits = sum(movie_id in RELEVANT_TEST_ITEMS for movie_id in recommended_ids)
    precision = precision_at_k(recommended_ids, RELEVANT_TEST_ITEMS, K_EVAL)
    recall = recall_at_k(recommended_ids, RELEVANT_TEST_ITEMS, K_EVAL)

    rows.append({
        "method": method,
        f"Hits@{K_EVAL}": hits,
        f"Precision@{K_EVAL}": precision,
        f"Recall@{K_EVAL}": recall,
    })

target_evaluation = pd.DataFrame(rows)
display(target_evaluation)

In [ ]:
# Reveal the relevant later interactions after you have calculated the metrics.
relevant_test_movies = (
    ratings_test[
        (ratings_test["userId"] == TARGET_USER)
        & (ratings_test["movieId"].isin(RELEVANT_TEST_ITEMS))
    ]
    .merge(movies[["movieId", "title", "genres"]], on="movieId", how="left")
    [["movieId", "title", "genres", "rating", "datetime"]]
    .sort_values("datetime")
)

display(relevant_test_movies)


### Question 5 - interpret one-user evaluation

**a.** Which method retrieved the most relevant later movies for this user?  
...

**b.** Does that prove this method is generally the best recommender? Why (not)?  
...

**c.** Why is this chronological evaluation more realistic than randomly hiding ratings from the user's history?  
...

**d.** If a recommended movie is not in `RELEVANT_TEST_ITEMS`, does that necessarily mean the user dislikes it? Explain.  
...


### a. Which method retrieved the most relevant later movies?
State the method with the highest Hits@10 in your evaluation table (e.g., Popularity, Content-based, or User-based CF).
### b. Does that prove this method is generally the best recommender? Why (not)?
No. Evaluating a single user is not statistically representative. User preferences vary widely, and performance must be aggregated across many users to evaluate general effectiveness.
### c. Why is chronological evaluation more realistic than randomly hiding ratings?
In production, recommenders predict future choices based on past behavior. A random split causes data leakage by using future interactions to predict past behavior.
### d. If a recommended movie is not in RELEVANT_TEST_ITEMS, does that necessarily mean the user dislikes it?
No. The test set only contains movies the user actually watched and rated. An unrated recommendation could be a movie the user would enjoy but has not yet discovered (an unobserved positive / false negative).

# 6. Predict the effect of changing one rating

Now test whether you understand what data each method uses.

Choose one movie that the target user rated **5 stars in the training data**. Imagine that the user had rated that same movie **1 star instead**.

## Predict without running anything

Complete this table:

| Method | Could its ranking change? | Why? |
|---|---|---|
| Popularity | ... | ... |
| Content-based | ... | ... |
| User-based CF | ... | ... |

# 7. Difficult situations

## Scenario A - new movie

A brand-new movie enters the catalogue. You know its **title and genres**, but **nobody has rated it yet**.

| Approach | Can meaningfully recommend the new movie immediately? | Why? |
|---|---|---|
| Popularity | No | Popularity scores count only the total number of interactions, regardless of rating value. The movie was already rated, so it remains filtered out |
| Content-based | Yes | The user profile averages genres from movies rated $\ge 4.0$. Dropping this movie to 1 star removes its genres from the profile, altering similarity scores |
| User-based CF | Yes | Neighbor similarity depends on absolute rating differences ($\vert{}r_u - r_v\vert{}$) on shared movies. Changing a 5 to a 1 shifts user distances, altering neighbor selection and predictions. |

> Note: for the earlier offline comparison, we deliberately restricted all methods to movies already known in training. Here we are explicitly asking about a **new item**.

## Scenario B - feedback loop and popularity bias

Suppose the platform repeatedly gives its highest-ranked movies much more exposure than other movies.

**How could this create a feedback loop and reinforce popularity bias?**

**YOUR ANSWER (2-3 sentences):**  
...

# 8. Final conclusion

Complete each statement in **one sentence**.

**Popularity-based recommendation is useful when...**  
...

**Content-based recommendation is useful when...**  
...

**Collaborative filtering is useful when...**  
...

Finally:

> What is **one thing you would investigate next** before deciding which recommender to deploy?

**YOUR ANSWER (1–2 sentences):**  
...


# Portfolio defence - be ready to explain

You do **not** need to write answers to these questions in advance, but you should be able to answer questions such as:

- Why is a recommender mainly a **ranking** problem here?
- Who are the users, items and interactions in MovieLens?
- Is the feedback implicit or explicit? Why?
- Why is the popularity baseline still not personalised after removing seen movies?
- What exactly does content-based filtering compare?
- What exactly does user-based collaborative filtering compare?
- Why did a particular movie appear in one Top-5 but not another?
- What do Precision@10 and Recall@10 mean in this notebook?
- Why do we train on earlier interactions and test on later ones?
- Why should you not conclude that one method is generally best from one user?
- What happens for a new movie with features but no ratings?
- How can recommendation exposure create a feedback loop and reinforce popularity bias?

## Submission checklist

Before submitting, check that:

- [ ] the notebook runs from top to bottom after you complete the TODOs
- [ ] `TARGET_USER` is set to an eligible user
- [ ] `target_history` is created with Python
- [ ] your popularity function returns only the Top-`n` unseen movies ranked by rating count
- [ ] all three Top-5 rankings are displayed side by side
- [ ] your target-user evaluation contains Hits@10, Precision@10 and Recall@10
- [ ] the cold-start and feedback-loop/popularity-bias questions are completed
- [ ] all required short-answer cells are completed
- [ ] you can explain the code and results during the portfolio defence